In [ ]:
#----------------------------#
# 0.) Idea
#----------------------------#
# Get PCA over comments data
#
# Approaches: 
# 1.) Extract embeddings Code
# 2.) PCA  Code
# 3.)Test Case
# 4.) Run
#
import sys
#import praw
#from praw.models import MoreComments
import pandas as pd
import numpy as np
#import re
from urlextract import URLExtract 
from datetime import datetime, timedelta
from collections import Counter
#from urlextract import URLExtract 
#from tldextract import extract
from IPython.core.display import display, HTML
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub
#import nltk
#from nltk.corpus import stopwords
#from nltk.tokenize import RegexpTokenizer
#from scipy import spatial
import import_ipynb
import math

In [ ]:
home_path = %pwd
data_path = "/home/pavel/Documents/Keusch_Job/Data/"
plot_path  = "/home/pavel/Documents/Keusch_Job/Tasks/subreddit_decide/"
subroutines_path = "/home/pavel/Documents/Keusch_Job/Subroutines/"

In [ ]:
#Subroutines
%cd  $subroutines_path
%run "selected_subreddits.ipynb"
%run "data_loader.ipynb"
%cd  $home_path

In [ ]:
#----------------------------#
# 1.) Extract embeddings Code
#----------------------------#

In [ ]:
# WARNING: download through http link takes almost 30minutes
# Thats why I have it on disc in the data path
#module_url = "https://tfhub.dev/google/universal-sentence-encoder/4" #@param ["https://tfhub.dev/google/universal-sentence-encoder/4", "https://tfhub.dev/google/universal-sentence-encoder-large/5"]
#model = hub.load(module_url)
#thx to https://stackoverflow.com/questions/50322001/how-to-save-load-a-tensorflow-hub-module-to-from-a-custom-path
module_url = data_path + "universal-sentence-encoder_4"
embed_model = hub.load(module_url)

In [ ]:
def embed(input): return embed_model(input)
def tf_simil(s1,s2): return np.inner(s1,s2) # inner Product as similarity! cool, but why inner product? 
# wouldn't you take cosine? 
def tf_simil_matrix(embeddings, rd = 8): return [[round(tf_simil(s1,s2),rd) for s1 in embeddings] for s2 in embeddings]

In [ ]:
#Don't need this one right now
def emb_dict_from_df(comment_df):
    emb_dict = {}
    for i in range(0,len(comment_df)):
        print(str(i) +  ":" + comment_df["id"].iloc[i])
        if isinstance(comment_df["body"].iloc[i], str):
            emb_dict[comment_df["id"].iloc[i]] = embed([comment_df["body"].iloc[i]])
        else:
            emb_dict[comment_df["id"].iloc[i]] = "NAN"
    return(emb_dict)

In [ ]:
#Old
#def emb_array_from_df(comment_df):
#    emb_matrix = np.append(embed([comment_df["body"].iloc[0]]),embed([comment_df["body"].iloc[1]]),axis=0)
#    for i in range(2,len(comment_df)):
#        print(str(i))
#        if isinstance(comment_df["body"].iloc[i], str):
#            emb_matrix = np.append(emb_matrix, embed([comment_df["body"].iloc[i]]),axis=0)
#        else:
#            na_vector = np.full((1,512),999)
#            emb_matrix = np.append(emb_matrix, na_vector,axis=0)
#    return(emb_matrix)

In [ ]:
def emb_array_from_df(comment_df):
    emb_matrix = np.append(embed([comment_df["body"].iloc[0]]),embed([comment_df["body"].iloc[1]]),axis=0)
    na_list = []
    for i in range(2,len(comment_df)):
        print(str(i))
        if isinstance(comment_df["body"].iloc[i], str):
            emb_matrix = np.append(emb_matrix, embed([comment_df["body"].iloc[i]]),axis=0)
        else:
            na_list.append(i)
    return(emb_matrix, na_list)

In [ ]:
#----------------------------#
# 2.) PCA  Code
#----------------------------#

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
MDS_embedding = MDS(n_components=2)
%matplotlib inline
import seaborn as sns

In [ ]:
#Also Old, since I right now don't use dicts
def dict_2_matrix(c_dict, key_list, print_iter = True):
    #s_dict: some dictionary where the values are embedings ()
    #key_list: a list of keys
    #RETURNS: the np.array matrix where each value is a row.
    emb_matrix = np.append(c_dict[key_list[0]], c_dict[key_list[1]],axis=0)
    for i in range(2,len(key_list)):
        if print_iter:
            print(str(i))
        emb_matrix = np.append(emb_matrix, c_dict[key_list[i]],axis=0)
    return(emb_matrix)

In [ ]:
def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x'] + 0.0002, point['y'], str(point['val']), alpha = 0.5)

In [ ]:
def emb_plot(emb_matrix, plot_path = plot_path , file_name = "subreddits_avg_emb_PCA_labels",
             label_series = selected_subreddits["subreddit"], main_title = "subreddits PCA",
             legend = True, plot_labels= False, MDS = False):
    
    # PCA
    if MDS:
        category_vectors = MDS_embedding.fit_transform(emb_matrix.T)
    else:
        model = PCA(n_components=2).fit(emb_matrix)
        category_vectors = model.components_.T
    #    # transform df
    category_vector_frame=pd.DataFrame(category_vectors, index = label_series.tolist(), #index=subr_list, 
                                   columns=['col1', 'col2']).reset_index()
    #
    #Plot 
    plt.figure(figsize=(10, 10))
    if legend:
        sns.scatterplot(data=category_vector_frame, x='col1', y='col2',hue='index', s = 20);
        plt.legend(loc='upper right')
    else:
        sns.scatterplot(data=category_vector_frame, x='col1', y='col2',s=20);
    if plot_labels:
        # need to do it because https://stackoverflow.com/questions/40339886/pandas-concat-generates-nan-values
        label_series_2 = label_series.copy()
        label_series_2.reset_index(drop=True, inplace=True)
        label_point(category_vector_frame.col1, category_vector_frame.col2,label_series_2, plt.gca())
    plt.title(main_title, size=20)
    plt.savefig(fname=plot_path + file_name)

In [ ]:
#----------------------------#
# 3.) Test Case
#----------------------------#

In [ ]:
#Too heavy, use sample
#C = load_data(data_name="COMM_2020_04.csv", data_path = data_path)
#conaservative_comments = C.loc[C["subreddit"] == "Conservative",]
#print(len(conservative_comments))
#del C

In [ ]:
C_sample = pd.read_csv(filepath_or_buffer = data_path +"C_sample.csv")
test_comments = C_sample.loc[C_sample["subreddit"] == "UpliftingNews",]
print(len(test_comments))
del C_sample

In [ ]:
#Testwise remove lines
remove_lines = [782,12,42,0]
remove_ids = test_comments["id"].iloc[remove_lines].tolist()
print(len(test_comments))
print(len(test_comments.loc[~test_comments["id"].isin(remove_ids),]))

In [ ]:
#Get embeddings
test_comments_matrix, remove_list = emb_array_from_df(test_comments)
test_comments_matrix = test_comments_matrix.T

In [ ]:
remove_list

In [ ]:
#test wise add some shit
print(remove_list)
remove_ids = test_comments["id"].iloc[remove_list].tolist()
print(len(test_comments))
test_comments = test_comments.loc[~test_comments["id"].isin(remove_ids),]
print(len(test_comments))

In [ ]:
emb_plot(test_comments_matrix,  plot_path = plot_path , file_name = "Test_Comments_PCA_sample",
        label_series = test_comments["body"], main_title ="Test PCA",
         legend = False, plot_labels= False,MDS = False)

In [ ]:
#----------------------------#
# 4.) Run
#----------------------------#

In [ ]:
#Old
#def processing_and_plot(subreddit_name, data_path = data_path, data_name = "COMM_2020_04.csv",
#                        plot_path = plot_path, 
#                        file_name = "Comments_PCA", main_title = "Comments PCA"):
#    # Load
#    C = load_data(data_name=data_name, data_path = data_path)
#    comments_df = C.loc[C["subreddit"] == subreddit_name,]
#    print("Loading "+ subreddit_name + " Rows: " + str(len(comments_df)))
#    if len(comments_df) < 1:
#        print("No Comments available")
#    else:
#        del C
#        ## Get Embeddinga
#        comments_emb_matrix = emb_array_from_df(comments_df)
#        comments_emb_matrix = comments_emb_matrix.T
#        #Plot
#        emb_plot(comments_emb_matrix,  plot_path = plot_path , file_name =file_name,
#                 label_series = comments_df["body"], main_title =main_title,
#                 legend = False, plot_labels= False,MDS = False)

In [ ]:
def processing_and_plot(subreddit_name, data_path = data_path, data_name = "COMM_2020_04.csv",
                        plot_path = plot_path,  save_matrix_path = data_path, 
                        file_name = "Comments_PCA", main_title = "Comments PCA",
                       save_matrix = False, plot=True):
    # Load
    C = load_data(data_name=data_name, data_path = data_path)
    if not isinstance(C, pd.DataFrame):
        C = pd.read_csv(filepath_or_buffer = data_path + data_name)
        if not isinstance(C, pd.DataFrame):
            return("No comparable Data found")
    comments_df = C.loc[C["subreddit"] == subreddit_name,]
    del C
    print("Loading "+ subreddit_name + " Rows: " + str(len(comments_df)))
    if len(comments_df) < 1:
        print("No Comments available")
    else:
        ## Get Embeddinga
        comments_emb_matrix, na_list = emb_array_from_df(comments_df)
        comments_emb_matrix = comments_emb_matrix.T
        # clean comments_df
        remove_ids = comments_df["id"].iloc[na_list].tolist()
        comments_df = comments_df.loc[~comments_df["id"].isin(remove_ids),]
        #Plot
        if plot:
            emb_plot(comments_emb_matrix,  plot_path = plot_path , file_name =file_name,
                     label_series = comments_df["body"], main_title =main_title,
                     legend = False, plot_labels= False,MDS = False)
        #Save Matric
        if save_matrix:
            np.save(save_matrix_path + subreddit_name + "_emb_matrix.npy", comments_emb_matrix)

In [ ]:
#processing_and_plot("Conservative", data_path = data_path, data_name = "COMM_2020_04.csv",
#                        plot_path = plot_path, 
#                        file_name = "Conservative_Comments_PCA",
#                        main_title = "Conservative Comments PCA")

In [ ]:
processing_and_plot("test", data_path = data_path, data_name = "COMM_2020_04.csv",
                        plot_path = plot_path,  save_matrix_path = data_path,
                        file_name = "Conservative_Comments_PCA",
                        main_title = "Conservative Comments PCA")

In [ ]:
subr_list = selected_subreddits["subreddit"].tolist()

In [ ]:
data_path + "comments_sample_matrices/"

In [ ]:
for s in subr_list:
    processing_and_plot(s, data_path = data_path, data_name = "C_sample.csv",
                        plot_path = plot_path, 
                        save_matrix_path = data_path + "comments_sample_matrices/" ,
                        file_name = s + "_Comments_PCA",
                        main_title = s + " Comments PCA",
                       save_matrix = True,
                       plot = False)

In [ ]:
# Kernel Died at Coronavirus

In [ ]:
# TO DO:
# S0: Titel on plots
# S1: Check out these sparse matrices
# S2: Structure and optimize code
#
# S3: Clustering
# S4: Regex resultate in Raum plotten
# S5: S1 und S2 overlap